In [36]:
from pathlib import Path
import json
import re
import pandas as pd

In [37]:
DATASET_ROOT = Path('~/Datasets/ADNI/ADNI/AD_NC').expanduser()
METADATA_PATH = Path('~/Datasets/ADNI/ADNI/meta_data_with_label.json').expanduser()

# get paths of each folder in dataset
FOLDERS = [
    DATASET_ROOT / 'train' / 'AD',
    DATASET_ROOT / 'train' / 'NC',
    DATASET_ROOT / 'test' / 'AD',
    DATASET_ROOT / 'test' / 'NC',
]

In [38]:
# open and read meta data json file
with METADATA_PATH.open('r', encoding='utf-8') as file:
    metadata = json.load(file)

PATIENT_PATTERN = re.compile(r"ADNI_(\d{3}_S_\d{4})", re.IGNORECASE)

def patient_id_from_record(record):
    value = record.get('raw')
    match = PATIENT_PATTERN.search(value)
    return match.group(1).upper()

scan_to_patient = {
    str(scan_id): patient_id_from_record(record)
    for scan_id, record in metadata.items()
}

print(f'Metadata records: {len(metadata)}')
print(f'Scans mapped to a patient: {len(scan_to_patient):}')
print(f'Unique patients in metadata: {len(set(scan_to_patient.values()))}')

Metadata records: 2189
Scans mapped to a patient: 2189
Unique patients in metadata: 942


In [ ]:
# get scan id from the file name of the image
def scan_id_from_filename(path):
    p = str(path)
    return p.split("/")[-1].split("_")[0]

rows = []

# iterate through dataset folder
for folder_path in FOLDERS:
    split = folder_path.parts[-2] # train or test
    for image_path in sorted(folder_path.rglob('*')):
        scan_id = scan_id_from_filename(image_path)
        record = metadata[scan_id]

        # add details related to image to list
        rows.append({
            'split': split,
            'image_name': image_path.name,
            'scan_id': scan_id,
            'patient_id': scan_to_patient.get(scan_id),
        })

# convert list to pandas dataframe
dataset_df = pd.DataFrame(rows)

print(f'Images Recorded: {len(dataset_df)}')
display(dataset_df.head())

Images Recorded: 30520


,split,image_name,scan_id,patient_id
0,train,218391_78.jpeg,218391,037_S_4001
1,train,218391_79.jpeg,218391,037_S_4001
2,train,218391_80.jpeg,218391,037_S_4001
3,train,218391_81.jpeg,218391,037_S_4001
4,train,218391_82.jpeg,218391,037_S_4001
